In [ ]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

In [31]:
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch
from tqdm import tqdm
import pandas as pd
import numpy as np

In [9]:
train_df = pd.read_csv("dataset/hm_train_df.csv")
train_df.head()

,customer_id,product_code,bought
0,0,0,2
1,0,1,2
2,0,2,1
3,0,3,1
4,0,4,1


In [17]:
val_df = pd.read_csv("dataset/hm_val_df.csv")
val_df.head()

,customer_id,product_code,bought
0,0,1389,1
1,0,3240,1
2,0,2767,1
3,0,846,1
4,0,411,1


In [18]:
n_users = train_df['customer_id'].nunique()
n_items = train_df['product_code'].nunique()

In [19]:
class HMDataset(Dataset):
    def __init__(self, x):
        self.x = x.reset_index(drop=True)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, index):

        row = self.x.iloc[index]
        user = torch.tensor(row.customer_id)
        item = torch.tensor(row.product_code)
        target = torch.tensor([row.bought]).float()

        return user,item,target

HMDataset(train_df)[0]

(tensor(0), tensor(0), tensor([2.]))

In [20]:
TensorDataset(torch.tensor(train_df.to_numpy()))[0]

(tensor([0, 0, 2]),)

In [21]:
def get_optimizer(net):
    optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

    return optimizer

In [24]:
import torch.nn as nn
import torch.nn.functional as F


#SVD model architecture
class HMModel(nn.Module):
    def __init__(self,n_users,n_items,k=30):
        super().__init__()
        torch.manual_seed(1)

        self.user_emb = nn.Embedding(n_users, embedding_dim=k).float()
        self.article_emb = nn.Embedding(n_items, embedding_dim=k).float()
        self.user_bias = nn.Embedding(n_users, embedding_dim=1).float()
        self.article_bias = nn.Embedding(n_items, embedding_dim=1).float()


    def forward(self, inputs):
        user_index, item_index = inputs[0], inputs[1]
        u_e = self.user_emb(user_index)
        i_e = self.article_emb(item_index)
        u_b = self.user_bias(user_index)
        i_b = self.article_bias(item_index)
        x = (u_e * i_e).sum(1) + u_b + i_b

        return x

In [25]:
#Models Initialization
model_dict = {}
for x in tqdm(range(n_users)):
    model_dict["model{0}".format(x)] = HMModel(n_users,n_items)

100%|█████████████████████████████████████| 1760/1760 [00:05<00:00, 342.51it/s]


In [26]:
model_dict

{'model0': HMModel(
   (user_emb): Embedding(1760, 30)
   (article_emb): Embedding(8618, 30)
   (user_bias): Embedding(1760, 1)
   (article_bias): Embedding(8618, 1)
 ),
 'model1': HMModel(
   (user_emb): Embedding(1760, 30)
   (article_emb): Embedding(8618, 30)
   (user_bias): Embedding(1760, 1)
   (article_bias): Embedding(8618, 1)
 ),
 'model2': HMModel(
   (user_emb): Embedding(1760, 30)
   (article_emb): Embedding(8618, 30)
   (user_bias): Embedding(1760, 1)
   (article_bias): Embedding(8618, 1)
 ),
 'model3': HMModel(
   (user_emb): Embedding(1760, 30)
   (article_emb): Embedding(8618, 30)
   (user_bias): Embedding(1760, 1)
   (article_bias): Embedding(8618, 1)
 ),
 'model4': HMModel(
   (user_emb): Embedding(1760, 30)
   (article_emb): Embedding(8618, 30)
   (user_bias): Embedding(1760, 1)
   (article_bias): Embedding(8618, 1)
 ),
 'model5': HMModel(
   (user_emb): Embedding(1760, 30)
   (article_emb): Embedding(8618, 30)
   (user_bias): Embedding(1760, 1)
   (article_bias): Emb

In [27]:
def read_data(data):
    return tuple(d for d in data[:-1]), data[-1]

In [28]:
read_data(HMDataset(train_df)[0])

((tensor(0), tensor(0)), tensor([2.]))

In [29]:
def validate(model_dict, val_loader):

    tbar = val_loader
    criterion = nn.MSELoss()
    loss_list = []
    for i in range(len(model_dict)):
        model_dict["model{0}".format(i)].eval()

    with torch.no_grad():
        for idx, data in enumerate(tbar):
            inputs, target = read_data(data)
            logits = model_dict["model{0}".format(int(inputs[0]))](inputs)
            loss = criterion(logits, target)

            loss_list.append(loss.detach().cpu().item())
        avg_loss = np.sqrt(np.mean(loss_list))

    return avg_loss

val_dataset = HMDataset(val_df)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [32]:
val_loss = []
optimizer_dict = {}
def train(model_dict, epochs):
    criterion = nn.MSELoss()
    val_map = validate(model_dict, val_loader)
    val_loss.append(val_map)
    for i in tqdm(range(n_users)):
        optimizer_dict["model{0}".format(i)] = get_optimizer(model_dict["model{0}".format(i)])

    for e in tqdm(range(epochs)):
      for i in range(len(model_dict)):
        model_dict["model{0}".format(i)].train()
      num_samples = 1387
      random_neighbors = np.random.choice(np.arange(1760), num_samples, replace=False)#Randomly select num_samples users to upload
      weights = []
      for n in random_neighbors:#Collect the partial weight of each selected user
        weights.append(len(split_train[n]))
      weights = np.array(weights) / sum(weights)
      #local training
      for neighbor in random_neighbors:#Only get the selected users' all data
        train_dataset = HMDataset(split_train[neighbor])
        train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

        local_epochs = 5
        for _ in range(local_epochs):
          tbar = train_loader
          for idx, data in enumerate(tbar):
              inputs, target = read_data(data)

              optimizer = optimizer_dict["model{0}".format(int(inputs[0]))]
              optimizer.zero_grad()
              logits = model_dict["model{0}".format(int(inputs[0]))](inputs)
              loss = criterion(logits, target)
              loss.backward()
              optimizer.step()
      #aggregation
      print("**Aggregation**")
      parameters = {}
      for i in range(len(random_neighbors)):
        neighbor_model = model_dict["model{0}".format(int(random_neighbors[i]))]
        for name,param in neighbor_model.named_parameters():

            if name not in parameters:
                parameters[name] = param.data * weights[i]
            else:
                parameters[name] = torch.add(parameters[name] * weights[i],param.data)

      #Update all users' paramters
      print('**Updating**')
      for i in range(len(model_dict)):
          neighbor_model = model_dict["model{0}".format(i)]
          for name,param in neighbor_model.named_parameters():

            param.data = parameters[name]

      val_map = validate(model_dict, val_loader)
      val_loss.append(val_map)
      log_text = f"Epoch {e+1}\nValidation Loss: {val_map}\n"
      print(log_text)


train(model_dict,epochs=20)

  0%|                                                                                                                                  | 0/20 [00:00<?, ?it/s]


NameError: name 'split_train' is not defined

In [ ]:
val_loss #Evaluation